# 📊 Evaluación de Recuperación RAG Híbrido con Reranker (FarmifAI)

Este notebook permite evaluar de forma rigurosa y cuantitativa el desempeño del pipeline **RAG Híbrido** optimizado para el proyecto **FarmifAI**:

- **Búsqueda Léxica:** BM25 con tokenización española robusta, filtrado de stopwords y stemming de **Snowball** (`nltk`).
- **Búsqueda Semántica:** Embeddings densos con **`intfloat/multilingual-e5-small`** utilizando esquema estricto de prefijos asimétricos (`passage: ` y `query: `).
- **Fusión Híbrida:** *Reciprocal Rank Fusion* (RRF) para combinar candidatos de ambas ramas.
- **Reranker de Alta Precisión:** Cross-Encoder multilingüe **`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`**.

### 🎯 Objetivos de la Evaluación:
1. **Hit Rate @ 20 (Unión Inicial):** Porcentaje de queries donde el chunk ideal (*ground truth*) fue recuperado por BM25 (top 10) o Búsqueda Semántica (top 10).
2. **Hit Rate @ 3 (Reranked - Ventana LLM):** Porcentaje de queries donde el chunk ideal quedó dentro del Top 3 final seleccionado para el contexto del LLM.
3. **Hit Rate @ 1 (Reranked - Máxima Relevancia):** Porcentaje de queries donde el chunk ideal ocupó la primera posición absoluta.
4. **Desglose de Origen (100% de los Queries):**
   - 📖 *Solo BM25 recuperó el chunk*
   - 🧠 *Solo Semántico recuperó el chunk*
   - ✨ *Ambos recuperaron el chunk*
   - ❌ *Ninguno recuperó el chunk (Miss @ 20)*
5. **Visualizaciones y Reporte:**
   - 📊 **Gráfica 1:** Distribución del 100% de los queries por mecanismo de recuperación.
   - 📉 **Gráfica 2:** Diagrama de embudo / barras de retención de relevancia ($100\% \rightarrow \text{Hit@20} \rightarrow \text{Hit@3} \rightarrow \text{Hit@1}$).
   - 📋 **Tabla Resumen y Exportación:** Resultados detallados por caso en CSV y JSONL con análisis de fallos.

---


## 1. Instalación de Dependencias


In [ ]:
# Instalación silenciosa de dependencias para RAG, métricas y visualización
!pip install -q rank-bm25 sentence-transformers nltk pandas matplotlib seaborn tqdm


## 2. Importaciones y Configuración del Entorno


In [ ]:
import json
import os
import pickle
import re
import time
import unicodedata
from typing import Any, Optional

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm.auto import tqdm

# Descargar recursos lingüísticos de NLTK
nltk.download('stopwords', quiet=True)

# Detectar si estamos en Google Colab
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("✅ Ejecutando en Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Ejecutando en entorno local")

# Configuración estética de gráficos para reportes de tesis
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

print("✅ Todas las librerías importadas correctamente.")


## 3. Carga de la Base de Conocimientos (`knowledge_base.json`) y Embeddings E5

Se carga la base de conocimiento estructurada de chunks agrícolas y sus embeddings semánticos precalculados con **`intfloat/multilingual-e5-small`** (`embeddings_chunks_e5_small.pkl`).

> [!IMPORTANT]
> El modelo **Multilingual-E5** requiere el prefijo asimétrico `passage: ` al indexar documentos y `query: ` al consultar para optimizar la representación vectorial.


In [ ]:
CHUNKS_FILENAME = "knowledge_base.json"
EMBEDDINGS_FILENAME = "embeddings_chunks_e5_small.pkl"
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"
RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"


def load_chunks(filename: str) -> list[dict]:
    """Carga los chunks desde un archivo JSON."""
    if not os.path.exists(filename):
        alt_paths = [os.path.basename(filename), f"output/JSON/{os.path.basename(filename)}", f"../../output/JSON/{os.path.basename(filename)}"]
        found = False
        for alt in alt_paths:
            if os.path.exists(alt):
                filename = alt
                found = True
                break
        if not found:
            if IN_COLAB:
                print(f"📂 Sube el archivo '{os.path.basename(filename)}':")
                uploaded = colab_files.upload()
                filename = os.path.basename(filename)
                if filename not in uploaded:
                    raise FileNotFoundError(f"No se subió el archivo '{filename}'.")
            else:
                raise FileNotFoundError(f"No se encontró '{filename}'.")

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    chunks = data["chunks"]
    print(f"✅ {len(chunks)} chunks cargados exitosamente desde '{filename}'.")
    return chunks


# --- Preprocesamiento BM25 Optimizado con Snowball Stemmer y Stopwords ---
stemmer = SnowballStemmer("spanish")

def remove_accents(text: str) -> str:
    """Remueve tildes y caracteres diacríticos."""
    text = unicodedata.normalize("NFD", text)
    return "".join(c for c in text if unicodedata.category(c) != "Mn")

# Pre-normalizar lista de stopwords en español para coincidencia insensible a tildes
spanish_stopwords = set(remove_accents(w.lower()) for w in stopwords.words("spanish"))

def preprocess_spanish(text: str) -> list[str]:
    """Preprocesa texto en español: minúsculas, sin acentos, sin stopwords y con stemming Snowball."""
    text = text.lower()
    text = remove_accents(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = [t for t in text.split() if len(t) > 1 and t not in spanish_stopwords]
    stemmed_tokens = [stemmer.stem(t) for t in tokens]
    return [t for t in stemmed_tokens if len(t) > 1]


# --- Embeddings Multilingual-E5 con Prefijo 'passage: ' ---
def load_or_generate_embeddings(
    chunks: list[dict],
    model_name: str,
    embeddings_file: str,
    batch_size: int = 128,
) -> tuple[np.ndarray, SentenceTransformer]:
    """Carga embeddings desde disco o los genera con prefijo 'passage: ' si no existen."""
    model = SentenceTransformer(model_name)
    print(f"✅ Modelo de embeddings cargado: {model_name}")

    embeddings = None
    if not os.path.exists(embeddings_file):
        alt_paths = [os.path.basename(embeddings_file), f"Notebooks/Colab/{os.path.basename(embeddings_file)}"]
        for alt in alt_paths:
            if os.path.exists(alt):
                embeddings_file = alt
                break

    if os.path.exists(embeddings_file):
        print(f"📦 Archivo de embeddings encontrado: '{embeddings_file}'")
        with open(embeddings_file, "rb") as f:
            embeddings = pickle.load(f)
        print(f"✅ Embeddings cargados desde disco: shape={embeddings.shape}")
    elif IN_COLAB:
        print(f"📂 ¿Tienes embeddings precalculados? Sube '{os.path.basename(embeddings_file)}' o cancela para generarlos.")
        try:
            uploaded = colab_files.upload()
            base_name = os.path.basename(embeddings_file)
            if base_name in uploaded:
                with open(base_name, "rb") as f:
                    embeddings = pickle.load(f)
                print(f"✅ Embeddings cargados desde subida: shape={embeddings.shape}")
        except Exception:
            print("ℹ️ No se subieron embeddings. Se generarán desde cero.")

    if embeddings is not None and (embeddings.shape[0] != len(chunks) or embeddings.shape[1] != 384):
        print(f"⚠️ Dimensión o cantidad no coincide: {embeddings.shape} vs ({len(chunks)}, 384). Regenerando...")
        embeddings = None

    if embeddings is None:
        # Multilingual-E5 requiere prefijo 'passage: ' para documentos
        texts_with_prefix = [f"passage: {chunk['text']}" for chunk in chunks]
        print(f"⏳ Generando embeddings Multilingual-E5 con prefijo 'passage: ' ({len(texts_with_prefix)} chunks, batch_size={batch_size})...")
        t0 = time.time()
        embeddings = model.encode(
            texts_with_prefix,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        print(f"✅ Embeddings generados en {time.time() - t0:.1f}s | shape={embeddings.shape}")
        with open(os.path.basename(embeddings_file), "wb") as f:
            pickle.dump(embeddings, f)
        print(f"💾 Embeddings guardados en '{os.path.basename(embeddings_file)}'")
        if IN_COLAB:
            colab_files.download(os.path.basename(embeddings_file))

    return embeddings, model


# Cargar base de conocimiento
chunks = load_chunks(CHUNKS_FILENAME)

# Tokenizar para BM25 con Snowball Stemmer
print("⏳ Tokenizando corpus para BM25 con Snowball Stemmer...")
corpus_tokens = [preprocess_spanish(c["text"]) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)
print(f"✅ Índice BM25 construido ({len(corpus_tokens)} documentos).")

# Cargar / Generar Embeddings Multilingual-E5
chunk_embeddings, embedding_model = load_or_generate_embeddings(
    chunks, EMBEDDING_MODEL_NAME, EMBEDDINGS_FILENAME
)

# Construir índice rápido para búsqueda de Ground Truth por texto
def normalize_text_lookup(text: str) -> str:
    return " ".join(text.strip().split())

text_to_chunk_idx = {c["text"].strip(): i for i, c in enumerate(chunks)}
norm_text_to_chunk_idx = {normalize_text_lookup(c["text"]): i for i, c in enumerate(chunks)}
print(f"✅ Mapeo de chunks a índices inicializado ({len(text_to_chunk_idx)} entradas).")


## 4. Carga y Parsing del Dataset de Evaluación (JSONL)

El dataset debe estar en formato **JSONL**, donde cada línea sigue la estructura de conversación definida en `format.json`:
```json
{"messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "<knowledge>\n{chunk_text}\n</knowledge>\n\n{pregunta_usuario}"},
    {"role": "assistant", "content": "..."}
]}
```
La función extraerá automáticamente la **pregunta de usuario** y el **chunk de conocimiento objetivo (Ground Truth)** para la evaluación.


In [ ]:
EVAL_DATASET_FILENAME = "dataset_agricola_eval.jsonl"


def load_eval_dataset(
    filename: str,
    chunks: list[dict],
    text_to_chunk_idx: dict[str, int],
    norm_text_to_chunk_idx: dict[str, int],
) -> list[dict[str, Any]]:
    """Carga y parsea el dataset JSONL de evaluación, vinculando el ground truth chunk."""
    if not os.path.exists(filename):
        alt_paths = [os.path.basename(filename), f"output/JSON/{os.path.basename(filename)}", f"../../output/JSON/{os.path.basename(filename)}"]
        found = False
        for alt in alt_paths:
            if os.path.exists(alt):
                filename = alt
                found = True
                break
        if not found:
            if IN_COLAB:
                print(f"📂 Sube el dataset de evaluación '{os.path.basename(filename)}':")
                uploaded = colab_files.upload()
                filename = os.path.basename(filename)
                if filename not in uploaded:
                    raise FileNotFoundError(f"No se subió el archivo '{filename}'.")
            else:
                raise FileNotFoundError(f"No se encontró '{filename}'.")

    knowledge_pattern = re.compile(r"<knowledge>\s*(.*?)\s*</knowledge>\s*(.*)", re.DOTALL)
    records = []
    unmatched = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                messages = data.get("messages", [])
                user_msg = next((m["content"] for m in messages if m.get("role") == "user"), "")
                assistant_msg = next((m["content"] for m in messages if m.get("role") == "assistant"), "")

                match = knowledge_pattern.search(user_msg)
                if not match:
                    print(f"⚠️ Línea #{line_idx + 1}: No se encontró etiqueta <knowledge>.")
                    continue

                gt_knowledge_text = match.group(1).strip()
                user_query = match.group(2).strip()

                # Buscar índice del chunk en la base de conocimientos
                chunk_idx = text_to_chunk_idx.get(gt_knowledge_text)
                if chunk_idx is None:
                    norm_k = normalize_text_lookup(gt_knowledge_text)
                    chunk_idx = norm_text_to_chunk_idx.get(norm_k)

                if chunk_idx is None:
                    unmatched += 1
                    print(f"⚠️ Línea #{line_idx + 1}: Chunk ground truth no encontrado en knowledge_base.json.")
                    continue

                gt_chunk = chunks[chunk_idx]
                records.append({
                    "eval_id": len(records) + 1,
                    "line_number": line_idx + 1,
                    "query": user_query,
                    "ground_truth_idx": chunk_idx,
                    "ground_truth_doc_id": gt_chunk.get("document_id", ""),
                    "ground_truth_chunk_number": gt_chunk.get("chunk_number", None),
                    "ground_truth_text": gt_knowledge_text,
                    "assistant_response": assistant_msg,
                })
            except json.JSONDecodeError as e:
                print(f"⚠️ Error decodificando JSON en línea #{line_idx + 1}: {e}")

    print(f"\n✅ Dataset cargado: {len(records)} preguntas válidas encontradas.")
    if unmatched > 0:
        print(f"⚠️ Advertencia: {unmatched} preguntas no pudieron vincularse al chunk de referencia.")
    else:
        print("🎯 100% de las preguntas fueron vinculadas exitosamente con su chunk ideal en el Knowledge Base.")

    return records


eval_dataset = load_eval_dataset(
    EVAL_DATASET_FILENAME, chunks, text_to_chunk_idx, norm_text_to_chunk_idx
)

# Muestra del primer elemento cargado
if eval_dataset:
    print("\n--- Ejemplo de Consulta de Evaluación #1 ---")
    print(f"Pregunta: {eval_dataset[0]['query']}")
    print(f"Ground Truth Doc: {eval_dataset[0]['ground_truth_doc_id']} | Chunk #{eval_dataset[0]['ground_truth_chunk_number']}")
    print(f"Ground Truth Preview: {eval_dataset[0]['ground_truth_text'][:120]}...")


## 5. Implementación del Pipeline Evaluador RAG

La clase `RAGRetrieverEvaluator` ejecuta y analiza cada paso del proceso:
1. **Búsqueda Léxica (BM25):** Recupera los $K_{lex}$ candidatos principales con preprocesamiento Snowball.
2. **Búsqueda Semántica (Multilingual-E5):** Codifica la consulta con prefijo `query: ` y calcula similitud coseno.
3. **Clasificación de Origen y Unión @ 20:** Identifica si el chunk ideal fue encontrado por:
   - 📖 **Solo BM25**
   - 🧠 **Solo Semántico**
   - ✨ **Ambos (Intersección)**
   - ❌ **Ninguno (Miss @ 20)**
4. **Fusión RRF y Reranker Cross-Encoder:** Evalúa los candidatos con `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` y verifica si el chunk ideal sobrevive en el **Top 3** (ventana para LLM) y en el **Top 1** (máxima relevancia).
5. **Métricas de Ranking y Tiempos:** Calcula posiciones exactas, Reciprocal Rank y latencias por componente.


In [ ]:
class RAGRetrieverEvaluator:
    """Evaluador integral del pipeline de recuperación RAG Híbrido + Reranker."""

    def __init__(
        self,
        chunks: list[dict],
        bm25_index: BM25Okapi,
        corpus_tokens: list[list[str]],
        chunk_embeddings: np.ndarray,
        embedding_model: SentenceTransformer,
        reranker_model_name: str = RERANKER_MODEL_NAME,
    ):
        self.chunks = chunks
        self.bm25 = bm25_index
        self.corpus_tokens = corpus_tokens
        self.chunk_embeddings = chunk_embeddings
        self.embedding_model = embedding_model

        print(f"⏳ Cargando modelo Cross-Encoder Reranker: '{reranker_model_name}'...")
        self.reranker = CrossEncoder(reranker_model_name)
        print(f"✅ Reranker listo: {reranker_model_name}")

    def search_bm25(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        """Búsqueda léxica con BM25 optimizado con Snowball Stemmer."""
        query_tokens = preprocess_spanish(query)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in top_indices]

    def search_semantic(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        """Búsqueda semántica densa con Multilingual-E5 (prefijo 'query: ')."""
        query_with_prefix = f"query: {query}"
        query_embedding = self.embedding_model.encode(
            query_with_prefix, normalize_embeddings=True, convert_to_numpy=True
        )
        similarities = self.chunk_embeddings @ query_embedding
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(int(i), float(similarities[i])) for i in top_indices]

    @staticmethod
    def reciprocal_rank_fusion(
        results_lists: list[list[tuple[int, float]]],
        k: int = 60,
    ) -> list[tuple[int, float]]:
        """Fusión de rankings por Reciprocal Rank Fusion (RRF)."""
        rrf_scores: dict[int, float] = {}
        for results in results_lists:
            for rank, (idx, _) in enumerate(results):
                rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
        sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results

    def rerank(
        self,
        query: str,
        candidate_indices: list[int],
    ) -> list[tuple[int, float]]:
        """Reordenamiento fino con Cross-Encoder (mmarco-mMiniLMv2)."""
        if not candidate_indices:
            return []
        pairs = [(query, self.chunks[i]["text"]) for i in candidate_indices]
        scores = self.reranker.predict(pairs)
        scored = list(zip(candidate_indices, [float(s) for s in scores]))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    def evaluate_query(
        self,
        query: str,
        gt_chunk_idx: int,
        top_k_lexical: int = 10,
        top_k_semantic: int = 10,
        top_k_final: int = 3,
    ) -> dict[str, Any]:
        """Evalúa una consulta individual a través de todo el pipeline."""
        # 1. Búsqueda Léxica
        t0 = time.time()
        bm25_res = self.search_bm25(query, top_k=top_k_lexical)
        t_bm25 = time.time() - t0
        bm25_indices = [idx for idx, _ in bm25_res]

        # 2. Búsqueda Semántica
        t0 = time.time()
        semantic_res = self.search_semantic(query, top_k=top_k_semantic)
        t_semantic = time.time() - t0
        semantic_indices = [idx for idx, _ in semantic_res]

        # 3. Análisis de Origen y Unión @ 20
        in_bm25 = gt_chunk_idx in bm25_indices
        in_semantic = gt_chunk_idx in semantic_indices
        hit_at_20 = in_bm25 or in_semantic

        if in_bm25 and in_semantic:
            origin_mechanism = "Ambos"
        elif in_bm25:
            origin_mechanism = "Solo BM25"
        elif in_semantic:
            origin_mechanism = "Solo Semántico"
        else:
            origin_mechanism = "Ninguno (Miss @ 20)"

        bm25_rank = (bm25_indices.index(gt_chunk_idx) + 1) if in_bm25 else None
        semantic_rank = (semantic_indices.index(gt_chunk_idx) + 1) if in_semantic else None

        # 4. Fusión RRF
        t0 = time.time()
        hybrid_res = self.reciprocal_rank_fusion([bm25_res, semantic_res])
        t_rrf = time.time() - t0
        hybrid_indices = [idx for idx, _ in hybrid_res]
        rrf_rank = (hybrid_indices.index(gt_chunk_idx) + 1) if gt_chunk_idx in hybrid_indices else None

        # 5. Reranking
        t0 = time.time()
        reranked_res = self.rerank(query, hybrid_indices)
        t_rerank = time.time() - t0
        reranked_indices = [idx for idx, _ in reranked_res]

        rerank_rank = (reranked_indices.index(gt_chunk_idx) + 1) if gt_chunk_idx in reranked_indices else None
        rerank_score = next((score for idx, score in reranked_res if idx == gt_chunk_idx), None)

        # 6. Métricas Finales de Hit Rate
        hit_at_3 = (rerank_rank is not None and rerank_rank <= top_k_final)
        hit_at_1 = (rerank_rank is not None and rerank_rank == 1)
        reciprocal_rank = (1.0 / rerank_rank) if rerank_rank is not None else 0.0

        # Top 3 chunks recuperados para inspección
        top_3_chunks = [
            {
                "rank": r + 1,
                "chunk_idx": idx,
                "document_id": self.chunks[idx].get("document_id", ""),
                "chunk_number": self.chunks[idx].get("chunk_number", None),
                "score": score,
                "is_ground_truth": (idx == gt_chunk_idx)
            }
            for r, (idx, score) in enumerate(reranked_res[:top_k_final])
        ]

        t_total = t_bm25 + t_semantic + t_rrf + t_rerank

        return {
            "query": query,
            "ground_truth_idx": gt_chunk_idx,
            "origin_mechanism": origin_mechanism,
            "hit_at_20": hit_at_20,
            "hit_at_3": hit_at_3,
            "hit_at_1": hit_at_1,
            "bm25_rank": bm25_rank,
            "semantic_rank": semantic_rank,
            "rrf_rank": rrf_rank,
            "rerank_rank": rerank_rank,
            "rerank_score": rerank_score,
            "reciprocal_rank": reciprocal_rank,
            "top_3_retrieved": top_3_chunks,
            "time_bm25_ms": t_bm25 * 1000,
            "time_semantic_ms": t_semantic * 1000,
            "time_rrf_ms": t_rrf * 1000,
            "time_rerank_ms": t_rerank * 1000,
            "time_total_ms": t_total * 1000,
        }

    def evaluate_dataset(
        self,
        eval_items: list[dict],
        top_k_lexical: int = 10,
        top_k_semantic: int = 10,
        top_k_final: int = 3,
        checkpoint_file: str = "eval_rag_checkpoint.jsonl",
    ) -> list[dict[str, Any]]:
        """Evalúa todo el dataset con barra de progreso y guardado incremental."""
        results = []
        print(f"\n🚀 Iniciando evaluación de {len(eval_items)} consultas...")
        print(f"   Configuración: Top BM25={top_k_lexical}, Top Semántico={top_k_semantic}, Top Final (LLM)={top_k_final}")

        checkpoint_f = open(checkpoint_file, "w", encoding="utf-8")

        try:
            for item in tqdm(eval_items, desc="Evaluando RAG"): 
                res = self.evaluate_query(
                    query=item["query"],
                    gt_chunk_idx=item["ground_truth_idx"],
                    top_k_lexical=top_k_lexical,
                    top_k_semantic=top_k_semantic,
                    top_k_final=top_k_final,
                )
                merged_res = {
                    "eval_id": item["eval_id"],
                    "line_number": item["line_number"],
                    "document_id": item["ground_truth_doc_id"],
                    "chunk_number": item["ground_truth_chunk_number"],
                    **res,
                }
                results.append(merged_res)
                checkpoint_f.write(json.dumps(merged_res, ensure_ascii=False) + "\n")
                checkpoint_f.flush()
        finally:
            checkpoint_f.close()

        print(f"✅ Evaluación completada. Checkpoint guardado en '{checkpoint_file}'.")
        return results


# Instanciar evaluador
evaluator = RAGRetrieverEvaluator(
    chunks=chunks,
    bm25_index=bm25,
    corpus_tokens=corpus_tokens,
    chunk_embeddings=chunk_embeddings,
    embedding_model=embedding_model,
    reranker_model_name=RERANKER_MODEL_NAME,
)


## 6. Ejecución del Benchmark de Evaluación

Configura los parámetros de evaluación y ejecuta el benchmark sobre las preguntas del dataset.


In [ ]:
# @title ⚙️ Configuración de la Evaluación
# @markdown Configura el tamaño de candidatos de primera etapa y la ventana final:

top_k_lexical = 10  # @param {type:"slider", min:3, max:30, step:1}
top_k_semantic = 10  # @param {type:"slider", min:3, max:30, step:1}
top_k_final = 3  # @param {type:"slider", min:1, max:10, step:1}
limitar_muestras = 0  # @param {type:"integer"}  # 0 para evaluar el 100% del dataset

samples_to_eval = eval_dataset if limitar_muestras <= 0 else eval_dataset[:limitar_muestras]
print(f"📊 Se evaluarán {len(samples_to_eval)} consultas.")

# Ejecutar evaluación
eval_results = evaluator.evaluate_dataset(
    samples_to_eval,
    top_k_lexical=top_k_lexical,
    top_k_semantic=top_k_semantic,
    top_k_final=top_k_final,
    checkpoint_file="eval_rag_checkpoint.jsonl",
)


## 7. Tabla Consolidada de Resultados y Métricas

Generamos el DataFrame detallado por caso y calculamos las métricas globales consolidadas:


In [ ]:
# Convertir a DataFrame
df_results = pd.DataFrame(eval_results)

# Métricas cuantitativas
total_queries = len(df_results)
hit_20_count = int(df_results['hit_at_20'].sum())
hit_3_count = int(df_results['hit_at_3'].sum())
hit_1_count = int(df_results['hit_at_1'].sum())

hit_20_pct = (hit_20_count / total_queries) * 100 if total_queries > 0 else 0
hit_3_pct = (hit_3_count / total_queries) * 100 if total_queries > 0 else 0
hit_1_pct = (hit_1_count / total_queries) * 100 if total_queries > 0 else 0

# Desglose por mecanismo de origen
cat_counts = df_results['origin_mechanism'].value_counts()
only_bm25 = int(cat_counts.get("Solo BM25", 0))
only_sem = int(cat_counts.get("Solo Semántico", 0))
both = int(cat_counts.get("Ambos", 0))
miss_20 = int(cat_counts.get("Ninguno (Miss @ 20)", 0))

mrr = float(df_results['reciprocal_rank'].mean())

# Latencias promedio (ms)
lat_bm25 = float(df_results['time_bm25_ms'].mean())
lat_sem = float(df_results['time_semantic_ms'].mean())
lat_rrf = float(df_results['time_rrf_ms'].mean())
lat_rerank = float(df_results['time_rerank_ms'].mean())
lat_total = float(df_results['time_total_ms'].mean())

# Construir tabla resumen
metrics_summary = [
    {"Métrica / Categoría": "Total de Consultas Evaluadas", "Conteo": total_queries, "Porcentaje (%)": "100.0%", "Descripción": "100% de los casos evaluados"},
    {"Métrica / Categoría": "🎯 Hit Rate @ 20 (Unión BM25 + Sem)", "Conteo": hit_20_count, "Porcentaje (%)": f"{hit_20_pct:.2f}%", "Descripción": "Chunk recuperado en primera etapa (candidatos)"},
    {"Métrica / Categoría": "🏆 Hit Rate @ 3 (Reranked - Ventana LLM)", "Conteo": hit_3_count, "Porcentaje (%)": f"{hit_3_pct:.2f}%", "Descripción": "Chunk presente en Top 3 final pasado al LLM"},
    {"Métrica / Categoría": "🥇 Hit Rate @ 1 (Reranked - Top 1)", "Conteo": hit_1_count, "Porcentaje (%)": f"{hit_1_pct:.2f}%", "Descripción": "Chunk clasificado en posición #1 absoluta"},
    {"Métrica / Categoría": "------------------------------------", "Conteo": "---", "Porcentaje (%)": "---", "Descripción": "------------------------------------"},
    {"Métrica / Categoría": "✨ Ambos (BM25 + Semántico)", "Conteo": both, "Porcentaje (%)": f"{(both / total_queries) * 100:.2f}%", "Descripción": "Recuperado por ambas ramas simultáneamente"},
    {"Métrica / Categoría": "🧠 Solo Semántico recuperó el chunk", "Conteo": only_sem, "Porcentaje (%)": f"{(only_sem / total_queries) * 100:.2f}%", "Descripción": "Recuperado exclusivamente por similitud semántica"},
    {"Métrica / Categoría": "📖 Solo BM25 recuperó el chunk", "Conteo": only_bm25, "Porcentaje (%)": f"{(only_bm25 / total_queries) * 100:.2f}%", "Descripción": "Recuperado exclusivamente por términos léxicos exactos"},
    {"Métrica / Categoría": "❌ Ninguno recuperó el chunk (Miss @ 20)", "Conteo": miss_20, "Porcentaje (%)": f"{(miss_20 / total_queries) * 100:.2f}%", "Descripción": "Chunk no encontrado en primera etapa"},
    {"Métrica / Categoría": "------------------------------------", "Conteo": "---", "Porcentaje (%)": "---", "Descripción": "------------------------------------"},
    {"Métrica / Categoría": "📈 MRR (Mean Reciprocal Rank)", "Conteo": f"{mrr:.4f}", "Porcentaje (%)": f"{mrr*100:.2f}%", "Descripción": "Promedio del inverso del rango final"},
    {"Métrica / Categoría": "⏱️ Latencia Promedio Total", "Conteo": f"{lat_total:.1f} ms", "Porcentaje (%)": "---", "Descripción": f"BM25: {lat_bm25:.1f}ms | Sem: {lat_sem:.1f}ms | Rerank: {lat_rerank:.1f}ms"},
]

df_metrics = pd.DataFrame(metrics_summary)

print("=" * 90)
print("📊 RESUMEN EJECUTIVO DE EVALUACIÓN RAG HÍBRIDO + RERANKER")
print("=" * 90)
display(df_metrics)

# Exportar archivos de resultados
csv_path = "eval_rag_results.csv"
jsonl_path = "eval_rag_results.jsonl"

df_export = df_results.drop(columns=["top_3_retrieved"]) if "top_3_retrieved" in df_results.columns else df_results
df_export.to_csv(csv_path, index=False, encoding="utf-8-sig")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for res in eval_results:
        f.write(json.dumps(res, ensure_ascii=False) + "\n")

print(f"\n💾 Resultados exportados exitosamente:")
print(f"   - CSV:   '{csv_path}'")
print(f"   - JSONL: '{jsonl_path}'")

if IN_COLAB:
    print("\n⬇️ Descargando CSV de resultados...")
    colab_files.download(csv_path)


## 8. 📊 Gráfica 1: Distribución del 100% de Queries por Fuente de Recuperación

Esta gráfica divide el **100% de las consultas** en cuatro categorías mutuamente excluyentes para ilustrar el aporte individual y combinado de cada técnica de búsqueda:
1. **Solo BM25 recuperó el chunk**
2. **Solo Semántico recuperó el chunk**
3. **Ambos recuperaron el chunk**
4. **Ninguno recuperó el chunk (Miss @ 20)**


In [ ]:
# Definir categorías y orden estándar
categories = ["Ambos\n(BM25 + Semántico)", "Solo Semántico\n(Rescate Semántico)", "Solo BM25\n(Rescate Léxico)", "Ninguno\n(Miss @ 20)"]
counts = [both, only_sem, only_bm25, miss_20]
percentages = [(c / total_queries) * 100 if total_queries > 0 else 0 for c in counts]

# Paleta de colores distintiva y profesional
colors = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c"]

# Crear figura
fig, ax = plt.subplots(figsize=(10, 5.5), dpi=150)

bars = ax.barh(categories, percentages, color=colors, edgecolor="#2c3e50", linewidth=1.2, height=0.55, alpha=0.9)

# Añadir etiquetas con conteo y porcentaje en cada barra
for bar, count, pct in zip(bars, counts, percentages):
    width = bar.get_width()
    ax.text(
        width + 1.2,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.2f}%  ({count:,} queries)",
        va="center",
        ha="left",
        fontsize=11,
        fontweight="bold",
        color="#2c3e50",
    )

# Ajustes visuales
ax.set_xlim(0, max(percentages) * 1.25 if max(percentages) > 0 else 100)
ax.set_xlabel("Porcentaje de Consultas (%)", fontsize=12, fontweight="bold", labelpad=10)
ax.set_title(
    "Distribución de Consultas por Mecanismo de Recuperación (100% de Casos)\nSistema RAG Híbrido FarmifAI",
    fontsize=14,
    fontweight="bold",
    pad=15,
    color="#1a252f",
)
ax.invert_yaxis()  # Mostrar 'Ambos' arriba
ax.grid(axis="x", linestyle="--", alpha=0.6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Resumen destacado en texto
hit_rate_union = hit_20_pct
plt.figtext(
    0.5,
    -0.04,
    f"[Resumen] Hit Rate @ 20 (Unión BM25 + Semántico): {hit_rate_union:.2f}% ({hit_20_count:,}/{total_queries:,} casos recuperados)",
    fontsize=11,
    fontweight="bold",
    ha="center",
    color="#1e8449",
)

plt.tight_layout()
plt.savefig("grafica_1_fuentes_recuperacion.png", bbox_inches="tight", dpi=300)
plt.show()
print("💾 Gráfica 1 guardada como 'grafica_1_fuentes_recuperacion.png'.")


## 9. 📉 Gráfica 2: Diagrama de Embudo y Retención de Relevancia del Pipeline

Esta gráfica muestra cómo se preserva la información relevante a medida que se reduce la ventana de contexto para el LLM:
$$\text{Consultas Totales (100\%)} \longrightarrow \text{Hit Rate @ 20 (Unión)} \longrightarrow \text{Hit Rate @ 3 (Reranked)} \longrightarrow \text{Hit Rate @ 1 (Reranked)}$$

Permite evaluar si el **Cross-Encoder Reranker** retiene exitosamente los chunks pertinentes en los primeros puestos.


In [ ]:
# Datos del embudo de retención
stages = [
    "1. Total Consultas\n(Dataset Completo)",
    "2. Hit Rate @ 20\n(Unión BM25 + Sem)",
    "3. Hit Rate @ 3\n(Reranked - Ventana LLM)",
    "4. Hit Rate @ 1\n(Reranked - Top 1)",
]

stage_counts = [total_queries, hit_20_count, hit_3_count, hit_1_count]
stage_percentages = [100.0, hit_20_pct, hit_3_pct, hit_1_pct]

# Paleta degradada de retención
funnel_colors = ["#2c3e50", "#2980b9", "#27ae60", "#16a085"]

fig, ax = plt.subplots(figsize=(11, 6.2), dpi=150)

x_pos = np.arange(len(stages))
bars = ax.bar(x_pos, stage_percentages, color=funnel_colors, edgecolor="#1a252f", linewidth=1.2, width=0.52, alpha=0.9)

# Añadir etiquetas en las barras (porcentaje y conteo)
for i, (bar, count, pct) in enumerate(zip(bars, stage_counts, stage_percentages)):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 2.0,
        f"{pct:.2f}%\n({count:,} q)",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="#1a252f",
    )

# Flechas de retención y badges entre etapas consecutivas
for i in range(len(stages) - 1):
    pct_prev = stage_percentages[i]
    pct_curr = stage_percentages[i + 1]
    retention_rate = (stage_counts[i + 1] / stage_counts[i]) * 100 if stage_counts[i] > 0 else 0
    x_mid = (x_pos[i] + x_pos[i+1]) / 2
    y_pos = min(pct_prev, pct_curr) / 2 + 10

    ax.annotate(
        f"Retención:\n{retention_rate:.1f}%",
        xy=(x_pos[i+1] - 0.26, pct_curr / 2 + 5),
        xytext=(x_mid, y_pos + 12),
        ha="center",
        va="center",
        fontsize=9.5,
        fontweight="bold",
        color="#b03a2e",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#fef9e7", edgecolor="#d35400", lw=1.2, alpha=0.95),
        arrowprops=dict(arrowstyle="->", color="#d35400", lw=1.5, connectionstyle="arc3,rad=-0.2"),
    )

# Configuración de ejes
ax.set_xticks(x_pos)
ax.set_xticklabels(stages, fontsize=11, fontweight="semibold")
ax.set_ylim(0, 122)
ax.set_ylabel("Retención de Relevancia (%)", fontsize=12, fontweight="bold", labelpad=10)
ax.set_title(
    "Embudo de Retención de Relevancia a Través del Pipeline RAG\n(Filtrado de Candidatos $\\rightarrow$ Cross-Encoder Reranker)",
    fontsize=14,
    fontweight="bold",
    pad=18,
    color="#1a252f",
)
ax.grid(axis="y", linestyle="--", alpha=0.6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("grafica_2_embudo_retencion.png", bbox_inches="tight", dpi=300)
plt.show()
print("💾 Gráfica 2 guardada como 'grafica_2_embudo_retencion.png'.")


## 10. 🔍 Diagnóstico e Inspección Cualitativa de Casos

Esta sección permite inspeccionar casos específicos para entender las fortalezas y debilidades del RAG:
- **Casos de rescate:** Consultas donde BM25 recuperó el chunk cuando la búsqueda semántica falló (y viceversa).
- **Casos de caída en Reranker:** Consultas donde el chunk estaba en el Top 20 pero el Cross-Encoder no lo dejó en el Top 3.
- **Misses totales:** Consultas donde ninguna de las dos ramas encontró el chunk en la primera etapa.


In [ ]:
# @title 🔎 Explorador Interactivo de Casos
# @markdown Selecciona el tipo de caso que deseas inspeccionar:

categoria_filtro = "Solo BM25"  # @param ["Solo BM25", "Solo Semántico", "Ambos", "Ninguno (Miss @ 20)", "En Top 20 pero fuera de Top 3"]
num_ejemplos = 3  # @param {type:"slider", min:1, max:10, step:1}

if categoria_filtro == "En Top 20 pero fuera de Top 3":
    subset = df_results[df_results["hit_at_20"] & (~df_results["hit_at_3"])]
else:
    subset = df_results[df_results["origin_mechanism"] == categoria_filtro]

print(f"\n🔎 Total de casos en '{categoria_filtro}': {len(subset)} ({len(subset)/total_queries*100:.2f}%)")
print("=" * 80)

for idx, row in subset.head(num_ejemplos).iterrows():
    print(f"\n📍 Caso #{row['eval_id']} (Línea {row['line_number']})")
    print(f"❓ Pregunta: {row['query']}")
    print(f"🏷️ Mecanismo: {row['origin_mechanism']}")
    print(f"📊 Ranks -> BM25: {row['bm25_rank']} | Semántico: {row['semantic_rank']} | RRF: {row['rrf_rank']} | Reranker: {row['rerank_rank']}")
    print(f"🎯 Hit@20: {'✅' if row['hit_at_20'] else '❌'} | Hit@3: {'✅' if row['hit_at_3'] else '❌'} | Hit@1: {'✅' if row['hit_at_1'] else '❌'}")
    
    # Obtener texto del chunk ground truth
    gt_text = chunks[row['ground_truth_idx']]['text']
    print(f"📖 Ground Truth Chunk (Doc: {row['document_id']}, #{row['chunk_number']}):")
    print(f"   \"{gt_text[:180]}...\"")
    print("-" * 80)
